# ATIVIDADE INTEGRADORA - RELATÓRIO OPERACIONAL DE PRÉ-DECOLAGEM

Quando falamos de telemetria de embarcações, principalmente foguetes, estamos tratando de alguns pontos críticos sobre o sucesso e fracasso da missão. Uma missão bem sucedida precisa iniciar com o pé direito, na decolagem.

Ao longo do desenvolvimento do processo da decolagem da missão Aurora 01, a decolagem precisaria ser extremamente bem sucedida, obtendo sucesso em cada mínimo detalhe, e para essa garantia de sucesso, precisaríamos de uma boa definição dos conceitos necessários para a execução da decolagem com segurança, minimizando os erros.

Além disso, a programação embarcada tem papel crucial para garantir que todos os sensores e módulos estejam funcionando corretamente, e no final garante o sucesso da decolagem.

## Organização da telemetria

O sucesso da missão depende totalmente da organização da telemetria e a definição clara das condições necessárias para a decolagem acontecer.

### Variáveis analisadas

O sucesso da missão depende das seguintes variáveis:
* Temperatura interna;
* Temperatura externa;
* Integridade estrutural (0/1);
* Níveis de energia (%);
* Pressão dos tanques;
* Status dos módulos críticos.

Dentre as variáveis, temos que definir inicialmente qual o tipo que elas assumirão.

No caso das temperaturas internas e externas, temos que assumirá valores numéricos definindo um valor da temperatura, nesse caso usaremos o sistema universal, e consideraremos em Celsius (ºC), dessa forma, as temperaturas irá assumir valores numéricos, como a variação milimétrica de temperatura possui pouca variação e pouca relevância, usaremos valores discretos para as variações de temperatura, então o tipo do dado assumirá valores inteiros (int).

Já para níveis de energia, temos uma carga total na bateria que é calculada para ter uma duração total da missão com um valor de sobra. Dessa forma, precisamos analisar como está a carga dela com base na carga total que ela suporta. Analisando esse conceito, iremos trabalhar com valores percentuais, fazendo a $carga_- atual/carga_-total$, dessa forma, por se tratar de resultado de uma divisão, tratamos o nivel de energia como valores numéricos com casas decimais (float).

Analisando os casos de pressão dos tanques, utilizaremos o sistema métrico como definição da análise da pressão (bar). Nesse método, como os sistemas precisam de métricas milimétricas referente a valores, usaremos para armazenar os dados de pressão os valores numéricos com casas decimais (float).

Os outros pontos como Status dos módulos críticos e Integridade Estrutural, o computador precisa realizar alguns testes para verificar se está íntegro ou se é preciso alguma manutenção em módulos e na integridade. Dessa forma, esses testes só podem receber dois valores do teste, True para caso a estrutura não apresente nenhum problema, e False caso apresente algum problema, assim, temos que esse tipo de variável são booleanos. Os booleanos também podem apresentar valores 0 e 1 para representar False e True respectivamente.

### Módulos Críticos

Quando falamos de verificar a integridade dos módulos críticos, temos que existe algumas características reais entre todos esses módulos. Entretanto, o sucesso da missão demanda conhecermos todos os nossos módulos críticos.

Precisamos reservar em algum lugar da memória um direcionamento dos nossos módulos, para isso, criaremos uma lista com nossos módulos críticos.

Os módulos críticos que consideraremos serão:
* O computador de voo (flight_computer);
* Sistema de navegação (navigation_system_status);
* Sistema de controle de atitude (attitude_control);
* Sistema de comunicação (communication_system);
* Sistema de energia (power_system);
* Sistema de propulsão (propulsion_system).

Cada módulo desse tem um papel importante no processo final de decisão, todos os sistemas precisam estar funcionais para que a missão seja um sucesso, dessa forma, todos os sistemas precisam estar funcionais antes de iniciar a decolagem. Dessa forma, precisamos testar todos os módulos para verificarmos se os módulos estão funcionais antes da decolagem.

Vamos iniciar alocando uma lista com todos os módulos críticos, quando algum módulo crítico precisar ser incluído na lista, basta appendarmos esse módulo na lista.


In [1]:
modulos_criticos = [ "flight_computer",
    "navigation_system",
    "attitude_control",
    "communication_system",
    "power_system",
    "propulsion_system"]

## Algoritmo de verificação
Criaremos um fluxograma de um algoritmo capaz de classificar se a decolagem está pronta para ser executada ou se precisaremos abortar a decolagem.

### Fluxograma


Como a análise dos módulos críticos são similares, precisamos receber valores booleanos a respeito dos dados, podemos tratar os dados referentes a esses módulos em uma classe. Com isso, criamos uma classe de módulos críticos, onde colocaremos as características comuns desses módulos.

In [102]:
import random

class ModulosCriticos:
    def __init__(self, nome):
      self.nome = nome
      self.status = None

    def __str__(self) -> str:
       return {self.nome:self.status}

    def testar_status(self):
        self.status = random.choices([True, False], weights=[0.9, 0.1])[0]
        return self.status

Usando a mesma lógica do uso de classes para organizarmos os pontos dos Módulos Críticos, usamos também uma classe para organizarmos a estrutura da telemetria. Nessa estrutura de classes, teremos as funções e os atributos que usaremos para a atividade.
Dentre os atributos temos:
* Temperatura Interna e externa: Considerados em °C
* Integridade Estrutural: Avaliado True se estiver integro e False se tiver falhas
* Voltagem bateria: Analisa a tensão da bateria
* Corrente da bateria: Analisa a corrente elétrica em Amperes
* Nivel Bateria: Porcentagem da bateria no momento da leitura
* Capacidade Bateria: Capacidade Total de armazenamento da bateria.
* Energia disponível: Cálculo com os dados para capturarmos a quantidade de energia disponível (kWh)
* Carga Potencia: Valor que vamos precisar durante o período da decolagem
* Perda Energia: Perda esperada de energia no processo da decolagem, avaliado em porcentagem
* Pressão Tanque: Analisa a pressão do momento no tanque
* Modulos: Análise da estrutura dos módulos descritos acima
* Dict Auditoria: Dicionário de motivos explicando cada campo no momento da análise (para auditoria).
* Dict Validações: Dicionário que uso para armazenar os valores das validações de cada campo obrigatório para decolagem.
* Dict Valores: Dicionário para armazenar os valores dos campos que calculamos.
* Decisão Decolagem: String resultante da decisão da telemetria de seguir com o lançamento ou abortar.

Além dos atributos, temos funções de definição de valor para cada campo. Iniciamos cada atributo no valor None e depois associamos os valores, uma vez que não faz sentido definirmos o valor dos atributos já no momento da inicialização da telemetria, visto que um módulo que em um momento inicial estava OK, pode se deteriorar ao longo do processo e não ser mais seguro realizar a decolagem com esse módulo, e devemos abortar a decolagem. Os outros componentes também podem sofrer alterações dos valores.

Além das funções de definição de valores, temos também as funções de validação, onde fazemos as validações se os campos estão em valores seguros para decolagem e autoriza a decolagem ou proíbe. Essas validações altera os valores dos atributos que são dicionários e da decisão de decolagem.

In [100]:
class Telemetria:
    def __init__(self, list_modulos):
        """
        Nessa camada do __init__ iniciamos todos os atributos que vamos precisar para
        o módulo da Telemetria.
        """
        self.temperatura_interna = None   # Possui
        self.temperatura_externa = None   # Possui
        self.integridade_estrutural = None  # Possui
        self.voltagem_bateria = None
        self.corrente_bateria = None
        self.nivel_bateria = None
        self.capacidade_bateria = None
        self.energia_disponivel = None
        self.carga_potencia = None
        self.perda_energia = None
        self.pressao_tanque = None
        self.modulos = [ModulosCriticos(modulo) for modulo in list_modulos]
        self.dict_validacoes = None
        self.dict_auditoria = None
        self.dict_valores = None
        self.decisao_decolagem = None

    @property
    def valores(self):
        return self.dict_valores

    # Funções para definir valores dos módulos
    def testar_todos_modulos(self):
        for modulo in self.modulos:
          modulo.testar_status()
        return self.modulos

    def captura_temperatura_interna(self):      # Está sendo calculada
        self.temperatura_interna = max(18, min(35, int(random.gauss(22,4))))   # Com essa distribuição, aumento as chances de cair um valor aleatório dentro do limite seguro de lançamento
        return self.temperatura_interna

    def captura_temperatura_externa(self):      # Está sendo calculada
        self.temperatura_externa = max(-10, min(35, int(random.gauss(12.5,22.5))))
        return self.temperatura_externa

    def captura_voltagem_bateria(self):       # Adicionada
        self.voltagem_bateria = random.uniform(46, 52)
        return self.voltagem_bateria

    def captura_corrente_bateria(self):       # Adicionada
        self.corrente_bateria = random.uniform(20, 120)
        return self.corrente_bateria

    def captura_energy_lvl(self):     # Está sendo calculada
        self.nivel_bateria = random.betavariate(5,1)    # Com essa distribuição aumento as chances de cair entre 80 e 100 de energia.
        return self.nivel_bateria

    def captura_capacidade_bateria(self):     # Adicionada
        self.capacidade_bateria = random.uniform(80, 120)
        return self.capacidade_bateria

    def captura_energia_disponivel(self):     # Adicionada
        self.energia_disponivel = (self.voltagem_bateria * self.capacidade_bateria * self.nivel_bateria) /1000
        return self.energia_disponivel

    def captura_carga_potencia(self):         # Adicionada
        self.carga_potencia = random.uniform(5, 25)
        return self.carga_potencia

    def captura_perda_energia(self):        # Adicionada
        self.perda_energia = random.uniform(2, 8)
        return self.perda_energia

    def captura_pressao_tanque(self):     # Está sendo calculada
        self.pressao_tanque = random.gauss(70,5)
        return self.pressao_tanque

    def captura_integridade_estrutural(self):   # Está sendo calculada
        self.integridade_estrutural = random.choices([True, False], weights=[0.9, 0.1])[0]
        return self.integridade_estrutural

    def captura_infos_telemetria(self):
        self.captura_temperatura_interna()
        self.captura_temperatura_externa()
        self.captura_integridade_estrutural()
        self.captura_energy_lvl()
        self.captura_pressao_tanque()

    def captura_infos_energia(self):
        self.captura_voltagem_bateria()
        self.captura_corrente_bateria()
        self.captura_capacidade_bateria()
        self.captura_carga_potencia()
        self.captura_energia_disponivel()
        self.captura_perda_energia()

    def define_valores_info_energia(self):
        self.captura_infos_energia()
        modulos_energia = ['voltagem_bateria', 'corrente_bateria', 'capacidade_bateria', 'carga_potencia', 'energia_disponivel', 'perda_energia']
        for modulo in modulos_energia:
            valor = getattr(self, modulo)
            self.dict_valores[modulo] = valor


    # Funções de validação
    def validacoes_telemetria(self, valores_seguros):
        self.testar_todos_modulos()
        self.captura_infos_telemetria()
        dict_validacoes = {}
        dict_auditoria = {}
        dict_valores = {}
        for sensor, valor_seguro in valores_seguros.items():
            valor = getattr(self, sensor)
            dict_valores[sensor] = valor
            if sensor == 'modulos':
                dict_validacoes[sensor] = all([modulo.status for modulo in valor])
                dict_valores[sensor] = {modulo.nome: modulo.status for modulo in valor}
                modulos_falhos = [modulo.nome for modulo in valor if not modulo.status]
                dict_auditoria[sensor] = {'valor_atual': all([modulo.status for modulo in valor]), 'regra': f'Os módulos {', '.join(modulos_falhos)}{" não " if len(modulos_falhos)!=0 else ""}estão seguros.'}

            elif isinstance(valor_seguro, dict):
                minimo = valor_seguro['min']
                maximo = valor_seguro['max']
                dict_validacoes[sensor] = minimo <= valor <= maximo
                dict_auditoria[sensor] = {'valor_atual': valor, 'regra': f'{sensor} deve estar entre {minimo} e {maximo}. Valor atual {valor}'}

            elif isinstance(valor_seguro, bool):
                dict_validacoes[sensor] = valor == valor_seguro
                dict_auditoria[sensor] = {'valor_atual': valor, 'regra': f'{sensor} deve ser {valor_seguro}.'}

        self.dict_validacoes = dict_validacoes
        self.dict_auditoria = dict_auditoria
        self.dict_valores = dict_valores
        self.define_valores_info_energia()

    def valida_decolagem(self):
        if all(self.dict_validacoes.values()):
            self.decisao_decolagem = "PRONTO PARA DECOLAR"
        else:
            self.decisao_decolagem = "DECOLAGEM ABORTADA"
        self.dict_valores['Decolagem'] = self.decisao_decolagem
        self.dict_valores['Motivo'] = self.dict_auditoria
        return self.decisao_decolagem


## Script em Python

Após fazermos a lógica por trás da Telemetria, usamos um dicionário para alimentarmos os valores seguros dos atributos necessários. Dessa forma, garantimos que não existe números mágicos dentro da execução do código e em caso de uma possível mudança nos valores seguros, conseguimos garantir a execução correta do código apenas alterando os valores desse dicionário, é importante esse campo não ser um atributo dentro da camada da classe Telemetria, visto que dependendo do local que vai ocorrer o lançamento, valores seguros para temperatura externa e pressão podem variar. Então, dessa forma, fazemos o código ser reutilizado. Repare que na função validacoes_telemetria onde validamos cada campo, recebemos o dicionário do valor seguro como um parâmetro.

In [104]:
valores_seguros = {
    "temperatura_interna": {"min": 18, "max": 30},
    "temperatura_externa": {"min": -10, "max": 35},
    "nivel_bateria": {"min": 0.8, "max": 1},
    "pressao_tanque": {"min": 60, "max": 80},
    "integridade_estrutural": True,
    "modulos": True
}

Finalizando todo a a definição dos valores dos campos e dos valores seguros, podemos realizar o teste da nossa Telemetria.

In [105]:
telemetria_aurora = Telemetria(modulos_criticos)
telemetria_aurora.validacoes_telemetria(valores_seguros)
telemetria_aurora.valida_decolagem()

'DECOLAGEM ABORTADA'

Podemos ver pelo código acima que a definição do status de autorizar ou proibir a decolagem está funcionando. Entretanto, em casos de abortar, não conseguimos identificar se a decolagem foi um falso positivo ou falso negativo. Pensando nisso, dentro da classe de Telemetria existe um encapsulamento dentro da camada valores, que servirá como um processo de auditoria para entendermos se os valores estão condizendo com o resultado que a função valida_decolagem associou.

In [106]:
telemetria_aurora.valores

{'temperatura_interna': 24,
 'temperatura_externa': -3,
 'nivel_bateria': 0.9016000473801543,
 'pressao_tanque': 65.38298417037038,
 'integridade_estrutural': False,
 'modulos': {'flight_computer': False,
  'navigation_system': False,
  'attitude_control': True,
  'communication_system': True,
  'power_system': True,
  'propulsion_system': True},
 'voltagem_bateria': 46.64471604371581,
 'corrente_bateria': 38.805159117649325,
 'capacidade_bateria': 96.96536123649577,
 'carga_potencia': 15.030808288226789,
 'energia_disponivel': 4.0778664559396605,
 'perda_energia': 3.870912782213793,
 'Decolagem': 'DECOLAGEM ABORTADA',
 'Motivo': {'temperatura_interna': {'valor_atual': 24,
   'regra': 'temperatura_interna deve estar entre 18 e 30. Valor atual 24'},
  'temperatura_externa': {'valor_atual': -3,
   'regra': 'temperatura_externa deve estar entre -10 e 35. Valor atual -3'},
  'nivel_bateria': {'valor_atual': 0.9016000473801543,
   'regra': 'nivel_bateria deve estar entre 0.8 e 1. Valor atua

## Análise energética

## Análise assistida por IA

O primeiro ponto para fazermos uma análise assistida com IA é criar um banco de ocorrências ao longo das execuções, salvaremos esses pontos em um dataframe. E precisamos de várias ocorrências desse modelo para conseguirmos analisar os pontos e